# 02 · Campaign — generate C3/C4/D2 assemblies + tied sequence design

**Standard slot:** *design campaign.* **For Project 04 this means:** generate **hundreds** of
symmetric backbones across **C3 / C4 / D2** with symmetric contigs, design each with **tied**
ProteinMPNN, and write `results/assemblies.csv` (D2). *Diversity before filtering* — generate
broadly now; triage in notebook 03.

Runs on the **mock** backend so it executes anywhere; the **real** symmetric RFdiffusion call is
shown (commented) with the A100 note. A free T4 realistically runs only a small C3 demo.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change)

Symmetric RFdiffusion, ColabDesign, and SymDesign move fast. Before relying on a pinned
version, confirm the upstream repo still exists. We pin a commit/tag in the comment; **update
the pin and log it** if the check fails or the API changed.

In [ ]:
import requests

# Pinned upstreams for this project — PIN A SPECIFIC COMMIT/TAG in your repo; tools change.
UPSTREAMS = {
    # RFdiffusion (symmetric mode). Pin example: a release tag or commit SHA you tested.
    "RFdiffusion": "https://github.com/RosettaCommons/RFdiffusion",   # pin a commit/tag
    # ColabDesign — symmetric RFdiffusion + AF2 wrappers + ProteinMPNN helpers.
    "ColabDesign": "https://github.com/sokrypton/ColabDesign",        # pin a commit/tag
    # SymDesign — symmetry definitions + symmetric docking concepts (optional).
    "SymDesign":  "https://github.com/kylemeador/symdesign",          # pin a commit/tag
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=20)
        print(f"{name:12s} {url}  ->  HTTP {r.status_code}")
    except Exception as e:
        print(f"{name:12s} {url}  ->  CHECK FAILED ({e!r}) — update the pin and log it")

## 1 · The campaign plan (symmetries × sizes)

We generate three symmetry families from `data/inputs/symmetry_defs.txt`. Scale `N_PER` up on
an A100; keep it tiny on a T4. The `mock` backend makes this loop run anywhere so you can build
the bookkeeping before paying for GPU.

In [ ]:
from sym_tools import generate_symmetric, tied_mpnn, SYMMETRY_ORDER
import pandas as pd

# (symmetry, subunit_length). Mirror data/inputs/symmetry_defs.txt; verify contigs for real runs.
CAMPAIGN = [("C3", 60), ("C4", 70), ("D2", 65)]
N_PER = 5          # backbones per family on the dry run; raise to 100s on an A100
N_SEQS = 4         # tied sequences per backbone
TOOL_GEN = "mock"  # -> "rfdiffusion" on an A100
TOOL_SEQ = "mock"  # -> "proteinmpnn" (tied) anywhere (MPNN is light)

for sym, L in CAMPAIGN:
    print(f"{sym}: {SYMMETRY_ORDER[sym]} subunits, subunit length {L}")

## 2 · The real symmetric RFdiffusion call (for the A100 run)

This is what the `tool="rfdiffusion"` backend wraps. **Verify the flags** against your pinned
RFdiffusion release — symmetric-mode flag names have changed across versions — and run it on an
**A100/HPC** for a full campaign.

In [ ]:
REAL_RFDIFFUSION_CALL = r"""
# A100 RECOMMENDED. Verify flags against the pinned RFdiffusion release before running.
./scripts/run_inference.py \
  inference.symmetry=C3 \
  'contigmap.contigs=[60-60]' \
  inference.num_designs=50 \
  inference.output_prefix=results/backbones/C3
# Then tied ProteinMPNN (light; CPU/T4 fine):
#   python protein_mpnn_run.py --pdb_path results/backbones/C3_0.pdb \
#     --tied_positions_jsonl tied.jsonl --num_seq_per_target 8 --sampling_temp 0.1
"""
print(REAL_RFDIFFUSION_CALL)

## 3 · Run the campaign (mock) → `results/assemblies.csv`

Generate backbones, design tied sequences for each, and record one row per (design × sequence).
Tied design means every symmetry-related subunit shares the sequence in that row. Switch the
`TOOL_*` flags to the real backends on an A100.

In [ ]:
rows = []
for sym, L in CAMPAIGN:
    backbones = generate_symmetric(sym, length=L, n=N_PER, tool=TOOL_GEN)
    for asm in backbones:
        seqs = tied_mpnn(asm, sym, n_seqs=N_SEQS, tool=TOOL_SEQ)
        for j, seq in enumerate(seqs):
            rows.append(dict(
                assembly_id=asm.assembly_id, seq_index=j, symmetry=sym,
                n_subunits=asm.n_subunits, subunit_length=L, contig=asm.contig,
                tied=True, gen_tool=TOOL_GEN, seq_tool=TOOL_SEQ, sequence=seq))
assemblies = pd.DataFrame(rows)
assemblies.to_csv("results/assemblies.csv", index=False)
print("wrote results/assemblies.csv", assemblies.shape)
print(assemblies.groupby("symmetry")["assembly_id"].nunique().rename("n_backbones"))
assemblies.head()

## 4 · An untied subset (for the tied-vs-untied benchmark in nb 04) `[extension]`

To ask whether tying actually helps, design a subset **without** tying (each chain free) and
tag it `tied=False`. In the mock backend we approximate this by perturbing the seed; on Colab,
run ProteinMPNN without the tied-positions JSON. Keep both in the pool for the comparison.

In [ ]:
# Untied subset: same backbones, tied=False. (Mock approximates by reusing tied_mpnn with a tag;
# on Colab, run ProteinMPNN WITHOUT the tied_positions_jsonl to get genuinely untied chains.)
untied_rows = []
for sym, L in CAMPAIGN[:1]:  # a subset (C3) is enough for the benchmark
    backbones = generate_symmetric(sym, length=L, n=N_PER, tool=TOOL_GEN)
    for asm in backbones:
        seqs = tied_mpnn(asm, sym, n_seqs=N_SEQS, tool=TOOL_SEQ)  # placeholder for an untied run
        for j, seq in enumerate(seqs):
            untied_rows.append(dict(
                assembly_id=asm.assembly_id + "_untied", seq_index=j, symmetry=sym,
                n_subunits=asm.n_subunits, subunit_length=L, contig=asm.contig,
                tied=False, gen_tool=TOOL_GEN, seq_tool=TOOL_SEQ, sequence=seq))
import pandas as pd
pool = pd.concat([assemblies, pd.DataFrame(untied_rows)], ignore_index=True)
pool.to_csv("results/assemblies.csv", index=False)
print("pool with tied + untied:", pool.shape, "| tied counts:")
print(pool["tied"].value_counts())

## D2 checklist
- [ ] `results/assemblies.csv`: C3/C4/D2, 100s of designs on the real run, one row per (design × sequence).
- [ ] Tied design verified (symmetry-related subunits share a sequence); an untied subset for the benchmark.
- [ ] Design log: gen/seq tool versions, RFdiffusion/ColabFold commit pins, seeds, contigs, runtimes in `LOG.md`.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared `oligomer` filter on your assemblies.